# XGBoost + LightGBM ensemble

**Data:** **`hf_data/02_feature_layer/training/outputs/`** or **`data/feature_data/02_feature_layer/training/outputs/`** (whichever contains CSVs) — loads the latest `hdb_feature_table_*.csv` and matching `hdb_feature_train_*` / `hdb_feature_test_*` with the same date suffix. Train/test CSVs are used to verify row counts match the full table.

**Target:** `resale_price`  
**Time column:** `transaction_year` (used for splits like `year` in 05b)

**Temporal split (same rule as `05b_xgb_lgb_ensemble.ipynb`):**
| Set | Rule on `transaction_year` | Role |
|-----|-----------------------------|------|
| **Train** | `< 2024` | Fit trees |
| **Validation** | `== 2024` | Early stopping + ensemble weight search |
| **Test** | `>= 2025` | Final held-out evaluation |

**Outputs:** `artifacts/` under this folder — models, weights, `feature_columns.json`.

In [25]:
%pip install -q numpy pandas pyarrow matplotlib scipy scikit-learn xgboost lightgbm joblib

Note: you may need to restart the kernel to use updated packages.


In [26]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
)
from sklearn.model_selection import TimeSeriesSplit
from scipy.optimize import minimize
import joblib

warnings.filterwarnings("ignore")

# ── Paths: cwd may be repo root, notebooks/, or 03_ml_layer_hybrid/ ──
HERE = Path.cwd().resolve()


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "feature_data").is_dir() or (p / "hf_data").is_dir():
            return p
    return start.parent


REPO_ROOT = _repo_root(HERE)

# Same locations as yc_hybrid_inference.default_feature_table_csv()
_FEATURE_OUTPUT_CANDIDATES = [
    REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs",
    REPO_ROOT / "data" / "feature_data" / "02_feature_layer" / "training" / "outputs",
]


def _feature_outputs_dir() -> Path:
    for d in _FEATURE_OUTPUT_CANDIDATES:
        if d.is_dir() and any(d.glob("hdb_feature_table_*.csv")):
            return d
    tried = "\n  ".join(str(d) for d in _FEATURE_OUTPUT_CANDIDATES)
    raise FileNotFoundError(
        "No hdb_feature_table_*.csv found. Run notebooks/00_download_data_from_HF.ipynb "
        "or place CSVs under one of:\n  " + tried
    )


HF_DATA_ROOT = _feature_outputs_dir()
OUT_DIR = HERE / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "resale_price"
YEAR_COL = "transaction_year"


def _latest_feature_snapshot_date(root: Path) -> str:
    """YYYYMMDD from latest hdb_feature_table_*.csv (train/test use same suffix)."""
    tables = sorted(root.glob("hdb_feature_table_*.csv"))
    if not tables:
        raise FileNotFoundError(f"No hdb_feature_table_*.csv under {root}")
    return tables[-1].stem.split("_")[-1]


_snap = _latest_feature_snapshot_date(HF_DATA_ROOT)
all_path = HF_DATA_ROOT / f"hdb_feature_table_{_snap}.csv"
train_path = HF_DATA_ROOT / f"hdb_feature_train_{_snap}.csv"
test_path = HF_DATA_ROOT / f"hdb_feature_test_{_snap}.csv"
for _p, _label in [(all_path, "table"), (train_path, "train"), (test_path, "test")]:
    if not _p.exists():
        raise FileNotFoundError(f"Missing {_label} file: {_p}")

print("HF_DATA_ROOT:", HF_DATA_ROOT)
print("Feature snapshot date:", _snap)
print("  table:", all_path.name)
print("  train:", train_path.name)
print("  test:", test_path.name)
print("OUT_DIR:", OUT_DIR)

HF_DATA_ROOT: /Users/bhuvesh/Documents/PropertyLens/data/feature_data/02_feature_layer/training/outputs
Feature snapshot date: 20260412
  table: hdb_feature_table_20260412.csv
  train: hdb_feature_train_20260412.csv
  test: hdb_feature_test_20260412.csv
OUT_DIR: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts


## 1 — Load full table (+ optional train/test CSV check)

CSV paths: `hf_data/.../outputs/` or `data/feature_data/.../outputs/` (see setup cell). The latest `hdb_feature_table_*.csv` date suffix is used for train and test files.

In [27]:
# all_path, train_path, test_path: latest hf_data snapshot (setup cell)

# Single source of truth for modelling (same rows as train+test CSVs combined)
df = pd.read_csv(all_path)
df_all = df  # alias for optional sections below

# Optional: verify pre-split files match the table
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
print("df (full table):", df.shape, "| years", df[YEAR_COL].min(), "–", df[YEAR_COL].max())
print("df_train file:  ", df_train.shape, "| years", df_train[YEAR_COL].min(), "–", df_train[YEAR_COL].max())
print("df_test file:   ", df_test.shape, "| years", df_test[YEAR_COL].min(), "–", df_test[YEAR_COL].max())
print("train+test rows == whole table?", len(df_train) + len(df_test) == len(df))

df (full table): (263004, 86) | years 2015 – 2026
df_train file:   (180195, 86) | years 2015 – 2022
df_test file:    (82809, 86) | years 2023 – 2026
train+test rows == whole table? True


## 2 — Feature columns (same idea as 05b)

Use all numeric columns except the target. Drop `address_key` (non-numeric / identifier). One-hot columns are already numeric/bool in this export.

In [28]:
META_COLS = ["address_key"]
# Keep transaction_year as a numeric feature (same role as `year` in 05b)
DROP_FROM_FEATURES = [TARGET] + META_COLS

def build_feature_columns(df):
    cols = [c for c in df.columns
            if c not in DROP_FROM_FEATURES and pd.api.types.is_numeric_dtype(df[c])]
    return cols

FEATURE_COLS = build_feature_columns(df)
assert set(FEATURE_COLS) == set(build_feature_columns(df_train)), "Table vs train CSV columns differ"
assert set(FEATURE_COLS) == set(build_feature_columns(df_test)), "Table vs test CSV columns differ"

print(f"Features: {len(FEATURE_COLS)}")

with open(OUT_DIR / "feature_columns.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print("Saved", OUT_DIR / "feature_columns.json")

Features: 84
Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/feature_columns.json


## 3 — Matrices: train/val from `df_train`, test from `df_test`

- `train_mask`: pre-2022 inside the training CSV  
- `val_mask`: 2022 inside the training CSV  
- `df_test`: held-out test file

In [29]:
def xy_from_df(frame):
    X = frame[FEATURE_COLS].fillna(0).astype(float)
    y = frame[TARGET].astype(float)
    return X, y

X = df[FEATURE_COLS].fillna(0).astype(float)
y = df[TARGET].astype(float)

train_mask = df[YEAR_COL] < 2024
val_mask = df[YEAR_COL] == 2024
test_mask = df[YEAR_COL] >= 2025

X_train, y_train = X.loc[train_mask].values, y.loc[train_mask].values
X_val, y_val = X.loc[val_mask].values, y.loc[val_mask].values
X_test, y_test = X.loc[test_mask].values, y.loc[test_mask].values

print(f"Train: {len(X_train):,} rows  (transaction_year < 2024)")
print(f"Val:   {len(X_val):,} rows  (transaction_year == 2024)")
print(f"Test:  {len(X_test):,} rows  (transaction_year >= 2025)")
print(f"\nMean {TARGET}: train ${y_train.mean():,.0f}  |  val ${y_val.mean():,.0f}  |  test ${y_test.mean():,.0f}")
print("Note: test mean often higher than train — typical for rising prices over time.")

Train: 205,930 rows  (transaction_year < 2024)
Val:   27,808 rows  (transaction_year == 2024)
Test:  29,266 rows  (transaction_year >= 2025)

Mean resale_price: train $481,273  |  val $612,616  |  test $653,132
Note: test mean often higher than train — typical for rising prices over time.


In [30]:
def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred) * 100),
        "r2": float(r2_score(y_true, y_pred)),
    }

def print_eval(d):
    print(
        f"  RMSE: ${d['rmse']:>10,.0f}  |  MAE: ${d['mae']:>10,.0f}  |  "
        f"MAPE: {d['mape']:.2f}%  |  R²: {d['r2']:.4f}"
    )

results = []

## 4 — TimeSeriesSplit CV on the fit portion (same structure as 05b)

In [31]:
tscv = TimeSeriesSplit(n_splits=5)
cv_rmses = []
for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train), 1):
    xgb_cv = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42,
        early_stopping_rounds=30, eval_metric="rmse", verbosity=0,
    )
    xgb_cv.fit(
        X_train[tr_idx], y_train[tr_idx],
        eval_set=[(X_train[va_idx], y_train[va_idx])], verbose=False,
    )
    pred = xgb_cv.predict(X_train[va_idx])
    rmse = np.sqrt(mean_squared_error(y_train[va_idx], pred))
    cv_rmses.append(rmse)
    print(f"  Fold {fold}: RMSE = ${rmse:,.0f}")

print(f"\nCV RMSE: ${np.mean(cv_rmses):,.0f} ± ${np.std(cv_rmses):,.0f}")

  Fold 1: RMSE = $35,509
  Fold 2: RMSE = $32,375
  Fold 3: RMSE = $38,886
  Fold 4: RMSE = $58,581
  Fold 5: RMSE = $45,635

CV RMSE: $42,197 ± $9,299


## 5 — Train XGBoost (early stopping on 2024 val)

In [32]:
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=8,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42,
    early_stopping_rounds=50, eval_metric="rmse", verbosity=0,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

r = evaluate("XGBoost", y_test, xgb_model.predict(X_test))
results.append(r)
print_eval(r)
joblib.dump(xgb_model, OUT_DIR / "xgb_model.joblib")
print("Saved", OUT_DIR / "xgb_model.joblib")

Training XGBoost...
  RMSE: $    85,221  |  MAE: $    70,934  |  MAPE: 10.60%  |  R²: 0.8262
Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/xgb_model.joblib


## 6 — Train LightGBM

In [33]:
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=800, learning_rate=0.05, num_leaves=63, max_depth=-1,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_model.fit(
    X_train, y_train, eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

r = evaluate("LightGBM", y_test, lgb_model.predict(X_test))
results.append(r)
print_eval(r)
joblib.dump(lgb_model, OUT_DIR / "lgb_model.joblib")
print("Saved", OUT_DIR / "lgb_model.joblib")

Training LightGBM...
  RMSE: $    89,488  |  MAE: $    74,180  |  MAPE: 11.03%  |  R²: 0.8084
Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/lgb_model.joblib


## 7 — Diversity check (validation residuals)

In [34]:
xgb_val_pred = xgb_model.predict(X_val)
lgb_val_pred = lgb_model.predict(X_val)
xgb_residuals = y_val - xgb_val_pred
lgb_residuals = y_val - lgb_val_pred
residual_corr = np.corrcoef(xgb_residuals, lgb_residuals)[0, 1]

print(f"Residual correlation (validation): {residual_corr:.4f}")
print(f"(1 - r) × 100% heuristic: {(1 - residual_corr) * 100:.1f}%")

xgb_better = np.abs(xgb_residuals) < np.abs(lgb_residuals)
print(f"XGB closer: {xgb_better.mean():.1%}  |  LGB closer: {(~xgb_better).mean():.1%}")

Residual correlation (validation): 0.9453
(1 - r) × 100% heuristic: 5.5%
XGB closer: 56.9%  |  LGB closer: 43.1%


## 8 — Optimise blend weights (MAPE on validation)

In [35]:
xgb_val = xgb_model.predict(X_val)
lgb_val = lgb_model.predict(X_val)
xgb_test = xgb_model.predict(X_test)
lgb_test = lgb_model.predict(X_test)

def blend_mape(weights):
    w = np.array(weights)
    pred = w[0] * xgb_val + w[1] * lgb_val
    return mean_absolute_percentage_error(y_val, pred) * 100

best_result = None
for _ in range(30):
    x0 = np.random.dirichlet(np.ones(2))
    res = minimize(blend_mape, x0=x0, method="Nelder-Mead", options={"maxiter": 5000})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

opt_weights = best_result.x
print(f"Optimal weights: XGB={opt_weights[0]:+.4f}, LGB={opt_weights[1]:+.4f}, sum={opt_weights.sum():.4f}")
print(f"Validation MAPE at optimal weights: {best_result.fun:.2f}%")

Optimal weights: XGB=+0.9559, LGB=+0.0900, sum=1.0459
Validation MAPE at optimal weights: 4.18%


## 9 — Ensemble on test set

In [36]:
ensemble_pred = opt_weights[0] * xgb_test + opt_weights[1] * lgb_test
r = evaluate("XGB + LGB Ensemble", y_test, ensemble_pred)
results.append(r)
print_eval(r)

xgb_r, lgb_r, ens_r = results[0], results[1], results[2]
print(f"\nΔ MAPE vs XGB: {xgb_r['mape'] - ens_r['mape']:+.2f} pp  |  vs LGB: {lgb_r['mape'] - ens_r['mape']:+.2f} pp")

  RMSE: $    63,367  |  MAE: $    47,308  |  MAPE: 7.00%  |  R²: 0.9039

Δ MAPE vs XGB: +3.60 pp  |  vs LGB: +4.03 pp


## 10 — Save ensemble artefacts (05b-style bundle)

In [37]:
joblib.dump(
    {
        "weights": opt_weights,
        "model_names": ["xgb", "lgb"],
        "feature_columns": FEATURE_COLS,
        "val_mape": best_result.fun,
        "yc_data": {
            "table_csv": str(all_path.name),
            "train_csv": str(train_path.name),
            "test_csv": str(test_path.name),
            "year_col": YEAR_COL,
            "split": "train < 2024 | val == 2024 | test >= 2025 (same as 05b)",
        },
        "description": "YC data: XGB + LGB weighted ensemble",
    },
    OUT_DIR / "ensemble_weights.joblib",
)
print("Saved", OUT_DIR / "ensemble_weights.joblib")

Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/ensemble_weights.joblib


## 11 — Optional: metrics on **all rows** of `df`

Same trained models evaluated on every row (including train/val/test). **Not** an honest generalization metric — use the test split above for that. Useful as a quick sanity check.

In [38]:
X_whole, y_whole = xy_from_df(df_all)
X_whole = X_whole.values
y_whole = y_whole.values

pred_x = xgb_model.predict(X_whole)
pred_l = lgb_model.predict(X_whole)
pred_e = opt_weights[0] * pred_x + opt_weights[1] * pred_l

print("Whole table:", len(y_whole), "rows")
print_eval(evaluate("XGBoost (whole)", y_whole, pred_x))
print_eval(evaluate("LightGBM (whole)", y_whole, pred_l))
print_eval(evaluate("Ensemble (whole)", y_whole, pred_e))

Whole table: 263004 rows
  RMSE: $    36,827  |  MAE: $    23,149  |  MAPE: 4.28%  |  R²: 0.9600
  RMSE: $    39,829  |  MAE: $    25,557  |  MAPE: 4.72%  |  R²: 0.9532
  RMSE: $    36,235  |  MAE: $    27,354  |  MAPE: 5.43%  |  R²: 0.9613
